# Lesson 22: Zava Retail Capstone (Part 3) — Testing, OpenTelemetry & Optimization

In this notebook, you will:
1. Run an automated **RLS & SQLi security test suite**.
2. Instrument Zava Retail queries with **OpenTelemetry** spans (`opentelemetry-api`, `opentelemetry-sdk`).
3. Compute $P_{50}/P_{95}$ latency and cache-hit ratios.

In [ ]:
!pip install -q -U opentelemetry-api opentelemetry-sdk

In [ ]:
import time
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("zava.retail.mcp")

with tracer.start_as_current_span("zava.execute_sales_query") as span:
    span.set_attribute("rls.store_id", "seattle")
    span.set_attribute("mcp.method", "tools/call")
    time.sleep(0.02)
    span.set_attribute("db.rows_returned", 2)

print("OpenTelemetry span exported successfully!")

### Challenge Exercise
Wrap the `ZavaDatabaseProvider` from Lesson 20 inside an OpenTelemetry span and assert that querying with `rls_store_id='seattle'` returns `0` rows belonging to `'redmond'`.